# Engenharia de features — Olist

Nesta etapa, organizo a base por pedido e preparo variáveis para prever atrasos no momento da aprovação do pagamento. A separação de treino e teste e o pipeline ficam para as próximas etapas.

A organização foi adaptada do projeto de referência utilizado no estudo. As decisões e verificações desta versão estão descritas abaixo; registre o link da referência na documentação do projeto.


In [ ]:
from pathlib import Path
import sys
import importlib
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

current = Path.cwd().resolve()
ROOT = next((p for p in [current, *current.parents]
             if (p / "module_olist").is_dir()
             and (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Abra o notebook dentro do projeto olist, com module_olist e pyproject.toml.")
sys.path.insert(0, str(ROOT))
import module_olist.dataset as dataset_module
import module_olist.features as features_module
importlib.reload(dataset_module)
importlib.reload(features_module)
from module_olist.dataset import load_dataset, create_dataset
from module_olist.features import create_features, FEATURES, TARGET

DATA = ROOT / "data" / "raw"
pd.set_option("display.max_columns", 100)


## 1. Montagem da base

Uso pedidos entregues com datas de aprovação, entrega real e entrega estimada preenchidas. O alvo mantém a comparação de timestamps usada na EDA: entrega real maior que entrega estimada significa atraso. Se o prazo for interpretado como o dia inteiro, essa regra precisará ser revista também na EDA.

Itens são agregados antes das junções. Pedidos sem itens ou estado permanecem visíveis para conferência. Esse recorte estuda atrasos entre pedidos entregues e não representa cancelamentos.


In [ ]:
orders, items, customers = load_dataset(
    DATA / "olist_orders_dataset.csv",
    DATA / "olist_order_items_dataset.csv",
    DATA / "olist_customers_dataset.csv",
)
base = create_dataset(orders, items, customers)
display(pd.DataFrame({"etapa": ["Pedidos originais", "Histórico analisável"],
                      "pedidos": [len(orders), len(base)]}))
display(base[TARGET].value_counts().reindex([0, 1], fill_value=0)
        .rename_axis("is_late").to_frame("pedidos"))
assert base["order_id"].is_unique, "Há pedidos repetidos."


## 2. Variáveis que vou investigar

| Grupo | Variáveis | Intenção |
|---|---|---|
| Prazo | promised_days, is_short_deadline | Medir o prazo após a aprovação; sete dias é uma hipótese de corte. |
| Tempo | approval_hours, purchase_month, purchase_weekday, purchase_hour, is_weekend_purchase | Descrever a aprovação e o momento da compra. |
| Pedido | item_count, seller_count, total_price, total_freight | Representar quantidade, vendedores e valores. |
| Relações | avg_item_price, freight_ratio | Separar valor médio por item e proporção do frete. |
| Localização | customer_state | Representar o estado do cliente. |

Preço e frete, isoladamente, não comprovam tamanho do produto ou distância. As variáveis serão avaliadas posteriormente.

Prazos não positivos, tempos de aprovação negativos e divisões com denominadores inválidos resultam em valores ausentes nos atributos calculados. Nenhuma linha é removida nesta função.


In [ ]:
refined = create_features(base)
assert len(refined) == len(base)
assert refined["order_id"].equals(base["order_id"])
display(refined[["order_id", *FEATURES, TARGET]].head())


## 3. Conferência dos resultados

As colunas futuras continuam na base histórica para auditoria, mas somente a lista FEATURES poderá entrar no modelo. Ela exclui entrega real, envio, status final e o alvo. Nenhuma mediana ou codificação é ajustada nesta etapa.


In [ ]:
future_columns = {"order_delivered_customer_date", "order_delivered_carrier_date",
                  "order_status", TARGET}
assert future_columns.isdisjoint(FEATURES)
missing = refined[FEATURES].isna().sum().rename("ausentes").to_frame()
missing["percentual"] = missing["ausentes"].div(len(refined) or 1).mul(100)
display(missing)
display(refined[FEATURES].describe(include="all").T)

diagnostics = pd.Series({
    "pedidos sem itens": base["item_count"].isna().sum(),
    "pedidos sem estado": base["customer_state"].isna().sum(),
    "prazo não positivo": ((base["order_estimated_delivery_date"] - base["order_approved_at"])
                            .dt.total_seconds().le(0)).sum(),
    "aprovação anterior à compra": (base["order_approved_at"] < base["order_purchase_timestamp"]).sum(),
    "quantidade não positiva": base["item_count"].le(0).sum(),
    "valor dos produtos não positivo": base["total_price"].le(0).sum(),
})
display(diagnostics.to_frame("pedidos"))


## 4. Registro da execução

Após executar com os CSVs, preencher:

- Quantos pedidos permaneceram no histórico?
- Quais atributos apresentaram valores ausentes ou inconsistências?
- Que tratamentos precisarão ser definidos na próxima etapa?

Esta etapa entrega funções reutilizáveis de integração e criação de variáveis. Ainda não é possível afirmar que uma variável melhora o modelo. O próximo passo será seguir a sequência de commits exigida para separação dos dados e modelagem.
